**IMPORT NECESSARY LIBRARIES**

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.cluster import KMeans
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    precision_recall_fscore_support,
    roc_auc_score,
    silhouette_score
)
import warnings
warnings.filterwarnings('ignore')

**LOAD AND PREPARE DATA**

In [ ]:
# Load the modeling dataset
df = pd.read_csv('student_data_modeling.csv')

**DISPLAY BASIC INFO**

In [ ]:
df.head()

,Student Number,Age,Gender,Residency,Class Standing,Scholarship,Study_Hours_Per_Week,Attendance_Rate,Y1_GWA,Y2_GWA,Y3_GWA,Y4_GWA,GPA_Trend,Final_GWA,Performance_Category
0,22-00001,20,Male,Dorm,Regular,Yes,31,97,1.77,1.58,1.54,1.55,Stable,1.63,Excellent
1,22-00002,23,Male,Family House,Regular,Yes,14,75,2.39,2.17,2.10,2.12,Stable,2.21,At-Risk
2,22-00003,24,Male,Family House,Regular,No,28,95,1.63,1.68,1.57,1.82,Stable,1.66,Good
3,22-00004,24,Male,Family House,Regular,No,34,91,1.79,1.62,1.47,1.77,Stable,1.66,Good
4,22-00005,23,Male,Dorm,Regular,No,27,82,1.88,1.85,1.72,2.02,Stable,1.85,Good


In [ ]:
df.describe()

,Age,Study_Hours_Per_Week,Attendance_Rate,Y1_GWA,Y2_GWA,Y3_GWA,Y4_GWA,Final_GWA
count,15000.000000,15000.00000,15000.000000,15000.000000,15000.000000,15000.000000,15000.000000,15000.000000
mean,21.973933,23.89540,88.289800,1.892690,1.851543,1.820036,1.850159,1.855727
std,1.412534,7.99149,7.298367,0.257789,0.252715,0.252298,0.277305,0.221254
min,20.000000,8.00000,70.000000,1.130000,1.140000,1.180000,1.070000,1.340000
25%,21.000000,18.00000,83.000000,1.700000,1.670000,1.620000,1.650000,1.700000
50%,22.000000,24.00000,89.000000,1.880000,1.830000,1.810000,1.850000,1.850000
75%,23.000000,30.00000,94.000000,2.110000,2.040000,1.990000,2.050000,2.010000
max,24.000000,40.00000,100.000000,2.570000,2.560000,2.610000,2.700000,2.410000


In [ ]:
df.columns

Index(['Student Number', 'Age', 'Gender', 'Residency', 'Class Standing',
       'Scholarship', 'Study_Hours_Per_Week', 'Attendance_Rate', 'Y1_GWA',
       'Y2_GWA', 'Y3_GWA', 'Y4_GWA', 'GPA_Trend', 'Final_GWA',
       'Performance_Category'],
      dtype='object')

In [ ]:
# Create a copy for modeling
df_model = df.copy()

**FEATURE ENGINEERING**

In [ ]:
# Encode binary categorical variables
df_model['Gender_Encoded'] = df_model['Gender'].map({'Male': 0, 'Female': 1})
df_model['Residency_Encoded'] = df_model['Residency'].map({'Family House': 0, 'Dorm': 1})
df_model['Class_Standing_Encoded'] = df_model['Class Standing'].map({'Regular': 0, 'Irregular': 1})
df_model['Scholarship_Encoded'] = df_model['Scholarship'].map({'No': 0, 'Yes': 1})
df_model['GPA_Trend_Encoded'] = df_model['GPA_Trend'].map({'Improving': 0, 'Stable': 1, 'Declining': 2})

In [ ]:
feature_columns = [
    'Age',
    'Gender_Encoded',
    'Residency_Encoded',
    'Class_Standing_Encoded',
    'Scholarship_Encoded',
    'Study_Hours_Per_Week',
    'Attendance_Rate',
    'Y1_GWA',
    'Y2_GWA',
    'Y3_GWA',
    'GPA_Trend_Encoded'
]

X = df_model[feature_columns].copy()

# Add realistic noise
# Increased noise to make data more realistic
np.random.seed(42)
X['Y1_GWA'] = X['Y1_GWA'] + np.random.normal(0, 0.25, len(X))
X['Y2_GWA'] = X['Y2_GWA'] + np.random.normal(0, 0.25, len(X))
X['Y3_GWA'] = X['Y3_GWA'] + np.random.normal(0, 0.22, len(X))
X['Study_Hours_Per_Week'] = X['Study_Hours_Per_Week'] + np.random.normal(0, 3, len(X))
X['Attendance_Rate'] = X['Attendance_Rate'] + np.random.normal(0, 5, len(X))

# Clip to valid ranges
X['Y1_GWA'] = X['Y1_GWA'].clip(1.0, 3.0)
X['Y2_GWA'] = X['Y2_GWA'].clip(1.0, 3.0)
X['Y3_GWA'] = X['Y3_GWA'].clip(1.0, 3.0)
X['Study_Hours_Per_Week'] = X['Study_Hours_Per_Week'].clip(5, 40)
X['Attendance_Rate'] = X['Attendance_Rate'].clip(60, 100)

y = df_model['Performance_Category']
student_ids = df_model['Student Number']

print(f"Selected {len(feature_columns)} features")
print(f"Features: {', '.join(feature_columns[:5])}...")

Selected 11 features
Features: Age, Gender_Encoded, Residency_Encoded, Class_Standing_Encoded, Scholarship_Encoded...


**SPLIT TRAIN AND TEST DATA**

In [ ]:
# Split data: 70% training, 30% testing
X_train, X_test, y_train, y_test, ids_train, ids_test = train_test_split(
    X, y, student_ids, test_size=0.30, random_state=42, stratify=y
)
print(f"Training set: {len(X_train):,} samples")
print(f"Testing set: {len(X_test):,} samples")

Training set: 10,500 samples
Testing set: 4,500 samples


**RANDOM FOREST MODEL**

In [ ]:
rf_model = RandomForestClassifier(
    n_estimators=200,
    max_depth=15,
    min_samples_split=10,
    min_samples_leaf=5,
    random_state=42,
    n_jobs=-1
)
rf_model.fit(X_train, y_train)

print("Model trained successfully")

Model trained successfully


In [ ]:
# Make predictions
print("\nMaking predictions...")
rf_predictions = rf_model.predict(X_test)
rf_probabilities = rf_model.predict_proba(X_test)
rf_confidence = rf_probabilities.max(axis=1)

# Evaluate performance
rf_accuracy = accuracy_score(y_test, rf_predictions)
rf_precision, rf_recall, rf_f1, _ = precision_recall_fscore_support(
    y_test, rf_predictions, average='weighted', zero_division=0
)


Making predictions...


**Evaluate Performance**

In [ ]:
print(f"\nRANDOM FOREST PERFORMANCE METRICS:")
print(f"   Accuracy:  {rf_accuracy*100:.2f}%")
print(f"   Precision: {rf_precision*100:.2f}%")
print(f"   F1-Score:  {rf_f1*100:.2f}%")



RANDOM FOREST PERFORMANCE METRICS:
   Accuracy:  85.33%
   Precision: 85.42%
   F1-Score:  85.10%


In [ ]:
# Confusion Matrix
rf_conf_matrix = confusion_matrix(y_test, rf_predictions)
print(f"\nCONFUSION MATRIX:")
print(f"{'─'*80}")
categories = sorted(y_test.unique())
print(f"\n{'':0} {'Predicted':>45}")
print(f"{'':15} {categories[0]:>15} {categories[1]:>15} {categories[2]:>15}")
print(f"{'─'*80}")
for i, actual in enumerate(categories):
    print(f"Actual {actual:10} {rf_conf_matrix[i][0]:>15} {rf_conf_matrix[i][1]:>15} {rf_conf_matrix[i][2]:>15}")


CONFUSION MATRIX:
────────────────────────────────────────────────────────────────────────────────

                                     Predicted
                        At-Risk       Excellent            Good
────────────────────────────────────────────────────────────────────────────────
Actual At-Risk                621               0             212
Actual Excellent                0             688             240
Actual Good                   111              97            2531


In [ ]:
# Feature Importance
print(f"\nTOP 10 MOST IMPORTANT FEATURES:")
print(f"{'─'*80}")
rf_feature_importance = pd.DataFrame({
    'Feature': feature_columns,
    'Importance': rf_model.feature_importances_
}).sort_values('Importance', ascending=False)

for idx, row in rf_feature_importance.head(10).iterrows():
    print(f"   {row['Feature']:30} {row['Importance']:.4f}")


TOP 10 MOST IMPORTANT FEATURES:
────────────────────────────────────────────────────────────────────────────────
   Study_Hours_Per_Week           0.2780
   Y3_GWA                         0.1945
   Y1_GWA                         0.1641
   Y2_GWA                         0.1586
   Attendance_Rate                0.1313
   Class_Standing_Encoded         0.0269
   Scholarship_Encoded            0.0150
   Age                            0.0130
   GPA_Trend_Encoded              0.0084
   Gender_Encoded                 0.0056


**GRADIENT BOOSTING MODEL**

In [ ]:
gb_model = GradientBoostingClassifier(
    n_estimators=50,
    learning_rate=0.05,
    max_depth=3,
    min_samples_split=30,
    min_samples_leaf=15,
    max_features='sqrt',
    random_state=42
)
gb_model.fit(X_train, y_train)

print("Model trained successfully")

Model trained successfully


In [ ]:
# Make predictions
print("\nMaking predictions...")
gb_predictions = gb_model.predict(X_test)
gb_probabilities = gb_model.predict_proba(X_test)
gb_confidence = gb_probabilities.max(axis=1)



Making predictions...


**Evaluate Performance**

In [ ]:
gb_accuracy = accuracy_score(y_test, gb_predictions)
gb_precision, gb_recall, gb_f1, _ = precision_recall_fscore_support(
    y_test, gb_predictions, average='weighted', zero_division=0
)

print(f"\nGRADIENT BOOSTING PERFORMANCE METRICS:")
print(f"Accuracy:  {gb_accuracy*100:.2f}%")
print(f"Precision: {gb_precision*100:.2f}%")
print(f"F1-Score:  {gb_f1*100:.2f}%")


GRADIENT BOOSTING PERFORMANCE METRICS:
Accuracy:  83.49%
Precision: 84.18%
F1-Score:  82.92%


In [ ]:
# Confusion Matrix
gb_conf_matrix = confusion_matrix(y_test, gb_predictions)
print(f"\nCONFUSION MATRIX:")
print(f"\n{'':0} {'Predicted':>45}")
print(f"{'':15} {categories[0]:>15} {categories[1]:>15} {categories[2]:>15}")
print(f"{'─'*80}")
for i, actual in enumerate(categories):
    print(f"Actual {actual:10} {gb_conf_matrix[i][0]:>15} {gb_conf_matrix[i][1]:>15} {gb_conf_matrix[i][2]:>15}")


CONFUSION MATRIX:

                                     Predicted
                        At-Risk       Excellent            Good
────────────────────────────────────────────────────────────────────────────────
Actual At-Risk                546               0             287
Actual Excellent                0             624             304
Actual Good                    73              79            2587


In [ ]:
# Feature Importance
print(f"\nTOP 10 MOST IMPORTANT FEATURES:")
print(f"{'─'*80}")
gb_feature_importance = pd.DataFrame({
    'Feature': feature_columns,
    'Importance': gb_model.feature_importances_
}).sort_values('Importance', ascending=False)

for idx, row in gb_feature_importance.head(10).iterrows():
    print(f"   {row['Feature']:30} {row['Importance']:.4f}")


TOP 10 MOST IMPORTANT FEATURES:
────────────────────────────────────────────────────────────────────────────────
   Study_Hours_Per_Week           0.3292
   Y3_GWA                         0.1838
   Y2_GWA                         0.1516
   Y1_GWA                         0.1408
   Attendance_Rate                0.1382
   Class_Standing_Encoded         0.0459
   Scholarship_Encoded            0.0055
   GPA_Trend_Encoded              0.0046
   Age                            0.0003
   Residency_Encoded              0.0001


**K-MEANS CLUSTERING**

**FEATURE SELECTION**

In [ ]:
cluster_feature_columns = [
    'Y1_GWA',
    'Y2_GWA',
    'Y3_GWA',
    'Y4_GWA',
    'Study_Hours_Per_Week',
    'Attendance_Rate',
    'GPA_Trend_Encoded'
]

X_cluster = df_model[cluster_feature_columns].copy()

# Add calculated average GWA (not the exact Final_GWA)
X_cluster['Avg_GWA'] = df_model[['Y1_GWA', 'Y2_GWA', 'Y3_GWA', 'Y4_GWA']].mean(axis=1)

# Standardize features (important for K-Means)
scaler = StandardScaler()
X_cluster_scaled = scaler.fit_transform(X_cluster)

print(f"Selected {len(cluster_feature_columns)+1} clustering features (including calculated Avg_GWA)")

Selected 8 clustering features (including calculated Avg_GWA)


In [ ]:
# Train K-Means model
print("\nTraining K-Means model with 3 clusters...")
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
cluster_labels = kmeans.fit_predict(X_cluster_scaled)


Training K-Means model with 3 clusters...


In [ ]:
# Calculate silhouette score
silhouette_avg = silhouette_score(X_cluster_scaled, cluster_labels)

print(f"Clustering complete")
print(f"\nCLUSTERING METRICS:")
print(f"Silhouette Score: {silhouette_avg:.4f}")

Clustering complete

CLUSTERING METRICS:
Silhouette Score: 0.3256


In [ ]:
import pandas as pd

# Add cluster labels to dataframe
df_model['Cluster'] = cluster_labels

# Analyze cluster profiles
print(f"\nCLUSTER PROFILES:")
print(f"{'─'*80}")

cluster_profiles = df_model.groupby('Cluster').agg({
    'Student Number': 'count',
    'Final_GWA': 'mean',
    'Study_Hours_Per_Week': 'mean',
    'Attendance_Rate': 'mean',
    'Y1_GWA': 'mean',
    'Y4_GWA': 'mean'
}).round(2)

cluster_profiles.columns = ['Count', 'Avg_Final_GWA', 'Avg_Study_Hours',
                           'Avg_Attendance', 'Avg_Y1_GWA', 'Avg_Y4_GWA']

print(f"{'Cluster':>8} {'Count':>8} {'Final GWA':>12} {'Study Hrs':>12} "
      f"{'Attendance':>12} {'Y1 GWA':>10} {'Y4 GWA':>10}")
print(f"{'─'*80}")

for cluster_id in range(3):
    profile = cluster_profiles.loc[cluster_id]
    print(f"{cluster_id:>8} {int(profile['Count']):>8} {profile['Avg_Final_GWA']:>12.2f} "
          f"{profile['Avg_Study_Hours']:>12.1f} {profile['Avg_Attendance']:>12.1f}% "
          f"{profile['Avg_Y1_GWA']:>10.2f} {profile['Avg_Y4_GWA']:>10.2f}")

# Interpret clusters
print(f"\nCLUSTER INTERPRETATION:")
print(f"{'─'*80}")

# Automatically name clusters based on characteristics
for cluster_id in range(3):
    profile = cluster_profiles.loc[cluster_id]
    avg_gwa = profile['Avg_Final_GWA']
    avg_study = profile['Avg_Study_Hours']
    avg_attend = profile['Avg_Attendance']
    y1_gwa = profile['Avg_Y1_GWA']
    y4_gwa = profile['Avg_Y4_GWA']

    # Determine cluster type
    if avg_gwa <= 1.7 and avg_study >= 25 and avg_attend >= 90:
        cluster_name = "Consistent High Performers"
    elif y4_gwa < y1_gwa - 0.2:
        cluster_name = "Improvement Seekers"
    elif avg_gwa >= 2.0 or avg_study < 20 or avg_attend < 85:
        cluster_name = "At-Risk/Intervention Needed"
    else:
        cluster_name = "Steady Performers"

    print(f"\n   Cluster {cluster_id}: {cluster_name}")
    print(f"   - Students: {int(profile['Count']):,}")
    print(f"   - Avg Final GWA: {avg_gwa:.2f}")
    print(f"   - Avg Study Hours: {avg_study:.1f} hrs/week")
    print(f"   - Avg Attendance: {avg_attend:.1f}%")


CLUSTER PROFILES:
────────────────────────────────────────────────────────────────────────────────
 Cluster    Count    Final GWA    Study Hrs   Attendance     Y1 GWA     Y4 GWA
────────────────────────────────────────────────────────────────────────────────
       0     3719         2.16         15.0         80.2%       2.17       2.18
       1     4168         1.59         33.0         95.7%       1.63       1.56
       2     7113         1.85         23.3         88.2%       1.90       1.85

CLUSTER INTERPRETATION:
────────────────────────────────────────────────────────────────────────────────

   Cluster 0: At-Risk/Intervention Needed
   - Students: 3,719
   - Avg Final GWA: 2.16
   - Avg Study Hours: 15.0 hrs/week
   - Avg Attendance: 80.2%

   Cluster 1: Consistent High Performers
   - Students: 4,168
   - Avg Final GWA: 1.59
   - Avg Study Hours: 33.0 hrs/week
   - Avg Attendance: 95.7%

   Cluster 2: Steady Performers
   - Students: 7,113
   - Avg Final GWA: 1.85
   - Avg Stu

In [ ]:
# ==============================================================================
# STEP 8: EXPORT RESULTS FOR POWER BI
# ==============================================================================
print("\n" + "="*80)
print("💾 EXPORTING RESULTS FOR POWER BI")
print("="*80)

print("\nExporting model predictions...")

# 1. Export predictions with confidence scores
predictions_df = pd.DataFrame({
    'Student_Number': ids_test.values,
    'Actual_Category': y_test.values,
    'RF_Predicted': rf_predictions,
    'GB_Predicted': gb_predictions,
    'RF_Confidence': (rf_confidence * 100).round(2),
    'GB_Confidence': (gb_confidence * 100).round(2)
})
predictions_df.to_csv('model_predictions.csv', index=False)
print("   ✓ model_predictions.csv")

# 2. Export feature importance comparison
feature_importance_df = pd.DataFrame({
    'Feature': feature_columns,
    'RF_Importance': rf_model.feature_importances_,
    'GB_Importance': gb_model.feature_importances_
}).sort_values('RF_Importance', ascending=False)
feature_importance_df.to_csv('feature_importance.csv', index=False)
print("   ✓ feature_importance.csv")

# 3. Export Random Forest confusion matrix
rf_cm_df = pd.DataFrame(
    rf_conf_matrix,
    index=[f'Actual_{cat}' for cat in categories],
    columns=[f'Pred_{cat}' for cat in categories]
)
rf_cm_df.to_csv('rf_confusion_matrix.csv')
print("   ✓ rf_confusion_matrix.csv")

# 4. Export Gradient Boosting confusion matrix
gb_cm_df = pd.DataFrame(
    gb_conf_matrix,
    index=[f'Actual_{cat}' for cat in categories],
    columns=[f'Pred_{cat}' for cat in categories]
)
gb_cm_df.to_csv('gb_confusion_matrix.csv')
print("   ✓ gb_confusion_matrix.csv")

# 5. Export Random Forest classification report
rf_report_df = pd.DataFrame(rf_class_report).transpose()
rf_report_df.to_csv('rf_classification_report.csv')
print("   ✓ rf_classification_report.csv")

# 6. Export Gradient Boosting classification report
gb_report_df = pd.DataFrame(gb_class_report).transpose()
gb_report_df.to_csv('gb_classification_report.csv')
print("   ✓ gb_classification_report.csv")

# 7. Export cluster assignments
cluster_export = df_model[[
    'Student Number', 'Cluster', 'Final_GWA', 'Study_Hours_Per_Week',
    'Attendance_Rate', 'Performance_Category', 'Y1_GWA', 'Y4_GWA'
]]
cluster_export.to_csv('cluster_assignments.csv', index=False)
print("   ✓ cluster_assignments.csv")

# 8. Export cluster profiles
cluster_profiles.to_csv('cluster_profiles.csv')
print("   ✓ cluster_profiles.csv")

# 9. Export model comparison metrics
comparison.to_csv('model_comparison.csv', index=False)
print("   ✓ model_comparison.csv")



💾 EXPORTING RESULTS FOR POWER BI

Exporting model predictions...
   ✓ model_predictions.csv
   ✓ feature_importance.csv
   ✓ rf_confusion_matrix.csv
   ✓ gb_confusion_matrix.csv
   ✓ rf_classification_report.csv
   ✓ gb_classification_report.csv
   ✓ cluster_assignments.csv
   ✓ cluster_profiles.csv
   ✓ model_comparison.csv
